# Step 1 — Extract Dataset from JPEG Images

Reads camera frames from a folder of JPEG images. No ROS installation needed.

**Output files (saved to `data/`):**
- `X_features.npy` — feature matrix, shape (N, 5)
- `y_labels.npy` — integer class labels (0=LEFT, 1=STRAIGHT, 2=RIGHT)

## Feature pipeline

$$
\text{BGR frame} \rightarrow \text{HSV mask (green)} \rightarrow \text{largest contour} \rightarrow \text{fitLine} \rightarrow [\text{offset, angle\_norm, area\_norm, aspect\_ratio, solidity}]
$$

## Features (5 total)

| # | Feature | Range | Meaning |
|---|---------|-------|---------|
| 1 | `offset` | −1 … +1 | horizontal position of line relative to image center |
| 2 | `angle_norm` | −1 … +1 | direction the line is heading (normalized) |
| 3 | `area_norm` | 0 … 1 | how much of the image is green line |
| 4 | `aspect_ratio` | 0 … 1 | shape of the line blob (tall=straight, wide=curve) |
| 5 | `solidity` | 0 … 1 | how solid/continuous the line is |

## Label scheme

| offset | Meaning | Label |
|--------|---------|-------|
| < −0.10 | Line left of centre | 0 = LEFT |
| −0.10 … +0.10 | Line centred | 1 = STRAIGHT |
| > +0.10 | Line right of centre | 2 = RIGHT |

## 1. Imports and paths

**Set `IMAGES_DIR` to the folder containing your JPEG files.**

In [ ]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

REPO_ROOT  = Path(r"C:/Users/danie/Desktop/Autonomous_Team4/GitBranch_Simulation")
IMAGES_DIR = REPO_ROOT / "Autonomous_Systems_A_Lab_Group_4" / "extracted_images"
DATA_DIR   = REPO_ROOT / "Autonomous_Systems_A_Lab_Group_4" / "data"
DATA_DIR.mkdir(exist_ok=True)

CLASS_NAMES = np.array(["LEFT", "STRAIGHT", "RIGHT"])

image_paths = sorted(
    list(IMAGES_DIR.glob("*.jpg")) + list(IMAGES_DIR.glob("*.jpeg")) + list(IMAGES_DIR.glob("*.JPG"))
)

assert len(image_paths) > 0, f"No JPEG files found in:\n{IMAGES_DIR}"
print(f"Images directory : {IMAGES_DIR}")
print(f"Total images     : {len(image_paths)}")
print(f"First image      : {image_paths[0].name}")
print(f"Last  image      : {image_paths[-1].name}")

## 2. Feature pipeline — HSV green masking + 5 geometric features

In [ ]:
# HSV range for the green line (BGR: 0, 255, 0)
LOWER_GREEN = np.array([40,  40,  40])
UPPER_GREEN = np.array([90, 255, 255])
MIN_CONTOUR_AREA = 100   # pixels — ignore tiny blobs
DEAD_BAND = 0.10         # offset threshold for LEFT/STRAIGHT/RIGHT


def extract_features(img_bgr: np.ndarray):
    """
    Extract 5 geometric features from a BGR frame using HSV green masking.

    Returns (feature_vector float32 shape (5,), offset float) or (None, None).

    Features:
      [0] offset       — line x position relative to center, in [-1, 1]
      [1] angle_norm   — line direction angle / 90, in [-1, 1]
      [2] area_norm    — contour area / image area, in [0, 1]
      [3] aspect_ratio — bounding box width / height, in [0, 1] (capped)
      [4] solidity     — contour area / convex hull area, in [0, 1]
    """
    rows, cols = img_bgr.shape[:2]

    # HSV green mask
    hsv  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, LOWER_GREEN, UPPER_GREEN)

    # Morphological cleanup
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    mask   = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask   = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)

    # Find largest green contour
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None, None

    contour = max(contours, key=cv2.contourArea)
    area    = cv2.contourArea(contour)
    if area < MIN_CONTOUR_AREA:
        return None, None

    # Feature 1 — offset: where is the line horizontally
    [vx, vy, x, y] = cv2.fitLine(contour, cv2.DIST_L2, 0, 0.01, 0.01)
    center_row = rows // 2
    if abs(float(vy)) > 0.01:
        line_x = float(x) + (center_row - float(y)) * (float(vx) / float(vy))
    else:
        M      = cv2.moments(contour)
        line_x = M["m10"] / M["m00"] if M["m00"] > 0 else cols / 2
    offset = float(np.clip((line_x - cols / 2.0) / (cols / 2.0), -1.0, 1.0))

    # Feature 2 — angle_norm: direction the line is heading
    angle      = float(np.degrees(np.arctan2(float(vy), float(vx))))
    angle_norm = float(np.clip(angle / 90.0, -1.0, 1.0))

    # Feature 3 — area_norm: how much of the image is green
    area_norm = float(np.clip(area / (rows * cols), 0.0, 1.0))

    # Feature 4 — aspect_ratio: shape of the bounding box
    _, _, bw, bh = cv2.boundingRect(contour)
    aspect_ratio  = float(np.clip(bw / (bh + 1e-6), 0.0, 1.0))

    # Feature 5 — solidity: how solid/continuous the line is
    hull     = cv2.convexHull(contour)
    hull_area = cv2.contourArea(hull)
    solidity  = float(np.clip(area / (hull_area + 1e-6), 0.0, 1.0))

    feat = np.array([offset, angle_norm, area_norm, aspect_ratio, solidity], dtype=np.float32)
    return feat, offset


def offset_to_label(offset: float) -> int:
    if offset < -DEAD_BAND:
        return 0  # LEFT
    elif offset > DEAD_BAND:
        return 2  # RIGHT
    return 1      # STRAIGHT


print("Feature pipeline defined.")
print(f"Dead-band: ±{DEAD_BAND}")
print("offset < -0.10  →  LEFT")
print("-0.10 to +0.10  →  STRAIGHT")
print("offset > +0.10  →  RIGHT")

## 3. Sanity check — single frame

In [ ]:
sample_img = cv2.imread(str(image_paths[0]))
feat, offset = extract_features(sample_img)

if feat is None:
    print("No green line detected in first frame — check IMAGES_DIR or HSV thresholds")
else:
    print(f"Feature vector : {feat}")
    print(f"offset={offset:+.3f}  label={CLASS_NAMES[offset_to_label(offset)]}")

    plt.figure(figsize=(8, 4))
    plt.imshow(cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB))
    plt.title(f"Sample frame — offset={offset:+.3f}")
    plt.axis("off")
    plt.show()

## 4. Extract features from all images

In [ ]:
SUBSAMPLE = 1   # 1 = use every image; 2 = every other image

X_list, offsets = [], []
skipped = 0
paths_to_use = image_paths[::SUBSAMPLE]
print(f"Processing {len(paths_to_use)} images (SUBSAMPLE={SUBSAMPLE})...")

for idx, img_path in enumerate(paths_to_use):
    img = cv2.imread(str(img_path))
    if img is None:
        skipped += 1
        continue

    feat, offset = extract_features(img)
    if feat is None:
        skipped += 1
        continue

    X_list.append(feat)
    offsets.append(offset)

    if (idx + 1) % 5000 == 0:
        print(f"  {idx+1}/{len(paths_to_use)} processed  ({len(X_list)} features extracted)")

print(f"\nDone. Extracted: {len(X_list)}  |  skipped: {skipped}")

## 5. Build labels and inspect distribution

In [ ]:
X       = np.array(X_list,  dtype=np.float32)
offsets = np.array(offsets, dtype=np.float32)
y       = np.array([offset_to_label(o) for o in offsets], dtype=np.int32)

print(f"Feature matrix X : {X.shape}  (N samples × 5 features)")
print(f"Labels y         : {y.shape}")
print()
print("Class distribution:")
for i, name in enumerate(CLASS_NAMES):
    mask   = y == i
    n      = mask.sum()
    mean_o = offsets[mask].mean() if n > 0 else 0.0
    print(f"  {name:10s}: {n:5d}  ({100*n/len(y):.1f}%)  mean offset={mean_o:+.3f}")

# Offset distribution
plt.figure(figsize=(8, 3))
plt.hist(offsets, bins=80, color="steelblue", edgecolor="none")
plt.axvline(-DEAD_BAND, color="red", linestyle="--", linewidth=1.5, label=f"±{DEAD_BAND} dead-band")
plt.axvline( DEAD_BAND, color="red", linestyle="--", linewidth=1.5)
plt.xlabel("offset (line position: −1=left, 0=centre, +1=right)")
plt.ylabel("Frame count")
plt.title("Line offset distribution")
plt.legend()
plt.tight_layout()
plt.show()

# Per-feature distributions
feat_names = ["offset", "angle_norm", "area_norm", "aspect_ratio", "solidity"]
fig, axes = plt.subplots(1, 5, figsize=(18, 3))
for i, (ax, name) in enumerate(zip(axes, feat_names)):
    ax.hist(X[:, i], bins=50, color="steelblue", edgecolor="none")
    ax.set_title(name)
    ax.set_xlabel("value")
axes[0].set_ylabel("count")
plt.suptitle("Feature distributions")
plt.tight_layout()
plt.show()

## 6. Save to data/

In [ ]:
np.save(str(DATA_DIR / "X_features.npy"), X)
np.save(str(DATA_DIR / "y_labels.npy"),   y)

print("Saved:")
for f in sorted(DATA_DIR.glob("*.npy")):
    print(f"  {f.name}  ({f.stat().st_size // 1024} KB)")
print("\nDone. Run 02_train_svm.ipynb next.")